<a href="https://colab.research.google.com/github/gianlucamajor/eCruziDB/blob/main/eda/agrs-eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Colab-**eCruzidb** Extended Data Analysis
## AGRs EDA

In [72]:
#@title Install dependencies and loading libs
%time
import os
from io import  StringIO
import pandas as pd
import requests
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
from scipy.stats import zscore


CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 4.77 µs


In [73]:
#@title Download data from eCruzidb
# The authority will not require after eCruzidb became open
%time
PE_json_url = "https://projetos.lbi.iq.usp.br/trypanosoma/ecruzidb/data/epitopes-data.json"
# PE_json_url = "https://projetos.lbi.iq.usp.br/trypanosoma/ecruzidb/data/control_and_chagasic_patients_mapped-segment-graph-graph-cc-id-msa-epitopes.json"

br_a4_chrs_url = "https://projetos.lbi.iq.usp.br/trypanosoma/ecruzidb/data/GCA_015033625.1-Br-A4-chromosomes.tsv"
user = "ecruzidb"
password = "setulab"

def PE_data_handler(data):
  df = pd.DataFrame(data)
  # Adjusting columns names
  df = df.rename(columns={"ID":"PE_ID",
                      "Number of Genomic Regions":"Number_of_AGRs",
                      "Number of Peptides":"Number_of_Peptides",
                      "Number of Inserts":"Number_of_Inserts",
                      "Number of Inserts by Group":"Number_of_Inserts_by_Clinical_Group",
                      "Epitope":"PE_Sequence",
                      "Genomic Region Locus":"Locus_of_AGRs"
                      }).copy()
  df['GAGR-ID'] = df['PE_ID'].apply(lambda x: x.split('-')[0]) # The GAGR could has zero, one or more PE.
  return df



# Predicted Epitope Data
PE_response = requests.get(PE_json_url, auth=(user, password))
if PE_response.status_code == 200:
    data = PE_response.json()
    PEs_df = PE_data_handler(data)

else:
    print(f"Failed to fetch data: {PE_response.status_code}")

# Br-A4 chromosomes data
br_a4_chrs_response = requests.get(br_a4_chrs_url, auth=(user, password))
if br_a4_chrs_response.status_code == 200:
    tsv_data = StringIO(br_a4_chrs_response.text)
    # original source: https://www.ncbi.nlm.nih.gov/datasets/genome/GCA_015033625.1/
    br_a4_chrs_complete_df = pd.read_csv(tsv_data, sep="\t")

else:
    print(f"Failed to fetch data: {br_a4_chrs_response.status_code}")



CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 5.01 µs


In [74]:
#@title Preprocessing chromosomes data: Building the pseudochromosome from unplaced scaffolds


def create_accession_and_contig_number(row):
  gbSeqAccession = row['GenBank seq accession']
  seqName = row['Sequence name']

  gbSeqAccessionNumber = int(gbSeqAccession.split('WNWZ')[1].split('.')[0])
  contigNumber = int(seqName.split('TcBrA4_Contig')[1])
  row['gbSeqAccessionNumber'] = gbSeqAccessionNumber
  row['contigNumber'] = contigNumber

  return row

# keep only useful columns
br_a4_chrs_df = br_a4_chrs_complete_df[['Chromosome name', 'GenBank seq accession', 'Seq length', 'Sequence name']].copy()

# select only unplacesd scaffolds
br_a4_scaffolds = br_a4_chrs_df[br_a4_chrs_df['GenBank seq accession'].str.startswith("WNWZ")].copy()
br_a4_scaffolds = br_a4_scaffolds.apply(lambda row: create_accession_and_contig_number(row), axis=1)

## Sorting the dataframe is fundamental for next steps
br_a4_scaffolds

br_a4_scaffolds.sort_values(by="gbSeqAccessionNumber", ascending=True, inplace=True)
br_a4_scaffolds.reset_index(drop=True, inplace=True)


## create a pseudochromosome to aggregate all unplaced scaffolds

start = 0
for i, row in br_a4_scaffolds.iterrows():
  # print(row['Seq length'])
  br_a4_scaffolds.loc[i, 'pse_chr_start'] = f"{start:d}"
  br_a4_scaffolds.loc[i, 'pse_chr_end'] = f"{start + row['Seq length']:d}"
  start = start + row['Seq length']

##

scaffold_id_and_start = {}
for i, row in br_a4_scaffolds.iterrows():
  scaffold_id_and_start[row['GenBank seq accession']] = row['pse_chr_start']

# scaffold_id_and_start

br_a4_scaffolds


,Chromosome name,GenBank seq accession,Seq length,Sequence name,gbSeqAccessionNumber,contigNumber,pse_chr_start,pse_chr_end
0,Un,WNWZ01000002.1,27086,TcBrA4_Contig101,1000002,101,0,27086
1,Un,WNWZ01000003.1,15041,TcBrA4_Contig102,1000003,102,27086,42127
2,Un,WNWZ01000004.1,15017,TcBrA4_Contig103,1000004,103,42127,57144
3,Un,WNWZ01000006.1,26761,TcBrA4_Contig105,1000006,105,57144,83905
4,Un,WNWZ01000007.1,26926,TcBrA4_Contig106,1000007,106,83905,110831
...,...,...,...,...,...,...,...,...
354,Un,WNWZ01000370.1,15106,TcBrA4_Contig98,1000370,98,8655383,8670489
355,Un,WNWZ01000383.1,26682,TcBrA4_Contig175,1000383,175,8670489,8697171
356,Un,WNWZ01000385.1,26168,TcBrA4_Contig331,1000385,331,8697171,8723339
357,Un,WNWZ01000386.1,103134,TcBrA4_Contig399,1000386,399,8723339,8826473


In [75]:
#@title Preprocessing chromosomes data:
br_a4_chrs_df = br_a4_chrs_complete_df[['Chromosome name', 'GenBank seq accession', 'Seq length']].copy()
br_a4_chrs_df.columns = ['chr', 'GBSeqAccession', 'SeqLength']
br_a4_chrs_df.set_index('GBSeqAccession', inplace=True)
br_a4_chrs_dict =  br_a4_chrs_df.to_dict(orient='index')

br_a4_chrs_dict

sectors = {}
for k in br_a4_chrs_dict.keys():
  chr = br_a4_chrs_dict.get(k).get('chr')
  length = br_a4_chrs_dict.get(k).get('SeqLength')
  if chr.isdigit(): # Only numeric (placed) chromosomes
    sectors[chr]=length

print(sectors)


{'1': 2738928, '2': 1986034, '3': 1768708, '4': 1676910, '5': 1492459, '6': 1421388, '7': 1369405, '8': 1336822, '9': 1155514, '10': 1097740, '11': 1076255, '12': 1041209, '13': 982025, '14': 975858, '15': 969620, '16': 927191, '17': 914771, '18': 909794, '19': 902532, '20': 846588, '21': 820352, '22': 815970, '23': 812063, '24': 778187, '25': 742617, '26': 731747, '27': 716856, '28': 711759, '29': 660991, '30': 660739, '31': 601716, '32': 590954, '33': 574917, '34': 285003, '35': 243420, '36': 231406, '37': 216495, '38': 166627, '39': 160921, '40': 155078, '41': 146158, '42': 141754, '43': 141550}


In [76]:
#@title Prepare dataframe
PEs_distinct_AGRs_df = PEs_df.drop_duplicates(subset=['GAGR-ID']).copy() # Keep just distinct AGRs Locus

def create_agrs_with_chr_info(agr_locus_list, chrs):
  agr_list_with_chr_info = []
  for agr in agr_locus_list:
    agr_splited = agr.split(':')
    chr_name = agr_splited[0]
    chr_start_end = agr_splited[1]
    # print(chr_start_end)
    chr = chrs.get(chr_name).get('chr')
    if chr.isnumeric():
      chr_gr = f"{chr}:{chr_start_end}"
    else:
      chr_gr = agr
    agr_list_with_chr_info.append(chr_gr)

  return agr_list_with_chr_info



# Create column with AGRs with numeric chr info
PEs_distinct_AGRs_df['AGRs_with_chr_info'] = PEs_distinct_AGRs_df.apply(lambda row: create_agrs_with_chr_info(row['Locus_of_AGRs'],br_a4_chrs_dict), axis=1)


## Show number of antigenic genomic regions used to precdict the 7,811 epitopes
unique_AGRs_set = set()
AGRs_list = []
for region_list in PEs_distinct_AGRs_df['AGRs_with_chr_info']:
  for region in region_list:
    r_splited = region.split(':')
    chr = r_splited[0]
    start_end = r_splited[1]

    start = start_end.split('-')[0]
    end = start_end.split('-')[1]
    AGRs_list.append([chr,start, end])
    unique_AGRs_set.add(region)

t = unique_AGRs_set
# display(sorted(genomic_regions))

all_antigenic_genomic_regions = pd.DataFrame(sorted(AGRs_list), columns=['chromosome', 'start', 'end'])
all_antigenic_genomic_regions


PEs_distinct_AGRs_df['Number_of_AGRs'].sum()


np.int64(21138)